# STAGE 9: Controlled Classical DDPG vs. Quantum QFRL Benchmark

**Objective**: Directly compare the Hybrid 4-Qubit Quantum Actor against a Classical MLP Actor under identical environment dynamics, hyperparameters, and seeds (`[42, 123, 2026]`).

**Hardware Required**: Kaggle GPU (Tesla T4) enabled.

In [ ]:
# Cell 1: Setup Working Directory and Clean Sync Repository
import os, sys, shutil

os.chdir('/kaggle/working')
repo_dir = '/kaggle/working/QRFL'

if os.path.exists(repo_dir):
    print('Syncing repository to latest origin/main...')
    %cd /kaggle/working/QRFL
    !git fetch origin
    !git reset --hard origin/main
    %cd /kaggle/working
else:
    print('Cloning fresh QRFL repository...')
    !git clone https://github.com/toqeer-ahmed/QRFL.git

qfrl_dir = os.path.join(repo_dir, 'QFRL_Highway')
os.chdir(qfrl_dir)
if qfrl_dir not in sys.path:
    sys.path.insert(0, qfrl_dir)

print(f'[READY] Working directory: {os.getcwd()}')


In [ ]:
# Cell 2: Install Quantum Backend and Support Libraries
!pip install -q --no-deps torchpack torchdiffeq
!pip install -q qiskit-ibm-runtime qiskit-aer loguru multimethod tensorpack toml
!pip install -q --no-deps git+https://github.com/mit-han-lab/torchquantum.git

# Verify imports
from rl.classical_actor import ClassicalDDPGAgent, ClassicalActor
from rl.ddpg import DDPGAgent
import torchquantum as tq

print(f'[PASS] All modules verified! TorchQuantum backend at: {tq.__file__}')


In [ ]:
# Cell 3: Launch Controlled Matched Benchmark
# Executes pilot smoke test and 15 episodes per seed across [42, 123, 2026] for BOTH models
!python -u experiments/compare_quantum_classical.py \
    --models classical quantum \
    --episodes 15 \
    --steps 50 \
    --seeds 42 123 2026 \
    --device cuda \
    --output_dir results


In [ ]:
# Cell 4: Export Results, Generate 1-Click ZIP Download, and Display Plots
import os, shutil
import pandas as pd
from IPython.display import Image, display, FileLink

src_results = '/kaggle/working/QRFL/QFRL_Highway/results'
dst_results = '/kaggle/working/results'
zip_output = '/kaggle/working/qfrl_stage9_results'

if os.path.exists(src_results):
    if os.path.exists(dst_results):
        shutil.rmtree(dst_results)
    shutil.copytree(src_results, dst_results)
    shutil.make_archive(zip_output, 'zip', src_results)
    
    print('\n' + '=' * 80)
    print('STAGE 9 BENCHMARK RUN COMPLETED SUCCESSFULLY!')
    print('=' * 80)
    print('⬇️ Click below to download Stage 9 package (CSV, plots, checkpoints):')
    display(FileLink('qfrl_stage9_results.zip'))
    
    plot_file = os.path.join(src_results, 'quantum_classical_comparison.png')
    if os.path.exists(plot_file):
        print('\n📈 Side-by-Side Comparison Learning Curves:')
        display(Image(filename=plot_file, width=900))
        
    csv_file = os.path.join(src_results, 'quantum_classical_comparison.csv')
    if os.path.exists(csv_file):
        print('\n📊 Metric Summary (First 10 rows):')
        df = pd.read_csv(csv_file)
        display(df[['model_type', 'seed', 'episode', 'energy_efficiency_mbits_per_joule', 'total_energy', 'episode_reward']].head(10))
else:
    print('[ERROR] Results folder not found.')
